In [146]:
import requests
import json
import pandas as pd
url = " http://api.nobelprize.org/v1/prize.json"
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36'
}

response = requests.get(url, headers=headers)
data = json.loads(response.content) 

print(type(data)) 

<class 'dict'>


In [147]:
prizes = data.get('prizes', [])
count = {}
for s in prizes:
        category = s.get('category')
        
        count[category] = count.get(category, 0) + 1
print(count)
        

{'chemistry': 125, 'economics': 57, 'literature': 125, 'peace': 125, 'physics': 125, 'medicine': 125}


In [148]:
df = pd.DataFrame(prizes)
print(df.head())

   year    category                                          laureates  \
0  2025   chemistry  [{'id': '1053', 'firstname': 'Susumu', 'surnam...   
1  2025   economics  [{'id': '1058', 'firstname': 'Joel', 'surname'...   
2  2025  literature  [{'id': '1056', 'firstname': 'László', 'surnam...   
3  2025       peace  [{'id': '1057', 'firstname': 'Maria Corina', '...   
4  2025     physics  [{'id': '1050', 'firstname': 'John', 'surname'...   

                                   overallMotivation  
0                                                NaN  
1  "for having explained innovation-driven econom...  
2                                                NaN  
3                                                NaN  
4                                                NaN  


In [149]:
pd.json_normalize(prizes)
df_flat = df.explode('laureates')
pd.json_normalize(df_flat['laureates']) 

,id,firstname,surname,motivation,share
0,1053,Susumu,Kitagawa,"""for the development of metal–organic frameworks""",3
1,1054,Richard,Robson,"""for the development of metal–organic frameworks""",3
2,1055,Omar M.,Yaghi,"""for the development of metal–organic frameworks""",3
3,1058,Joel,Mokyr,"""for having identified the prerequisites for s...",2
4,1059,Philippe,Aghion,"""for the theory of sustained growth through cr...",4
...,...,...,...,...,...
1070,569,Sully,Prudhomme,"""in special recognition of his poetic composit...",1
1071,462,Henry,Dunant,"""for his humanitarian efforts to help wounded ...",2
1072,463,Frédéric,Passy,"""for his lifelong work for international peace...",2
1073,1,Wilhelm Conrad,Röntgen,"""in recognition of the extraordinary services ...",1


In [151]:
# in halghe row haie khali ro tashkhis mide = [d for d in prizes if 'laureates' in d and d ['laureates'] is not None]
prizes_list_clean = [d for d in prizes if 'laureates' in d and d['laureates'] is not None]

df_last = pd.json_normalize(data=prizes_list_clean, record_path='laureates', meta=['category', 'year'])

print(df_last.head())

     id firstname   surname  \
0  1053    Susumu  Kitagawa   
1  1054   Richard    Robson   
2  1055   Omar M.     Yaghi   
3  1058      Joel     Mokyr   
4  1059  Philippe    Aghion   

                                          motivation share   category  year  
0  "for the development of metal–organic frameworks"     3  chemistry  2025  
1  "for the development of metal–organic frameworks"     3  chemistry  2025  
2  "for the development of metal–organic frameworks"     3  chemistry  2025  
3  "for having identified the prerequisites for s...     2  economics  2025  
4  "for the theory of sustained growth through cr...     4  economics  2025  


In [153]:
michael = df_last[df_last['firstname'].str.contains('Michael' , na=False , case=False)]
print(f"number of laureates with the name Michael: {len(michael)}")
print(michael[['firstname', 'surname','year']].head())

number of laureates with the name Michael: 10
      firstname     surname  year
74      Michael    Houghton  2020
81      Michael      Kremer  2019
113     Michael     Rosbash  2017
114  Michael W.       Young  2017
124  J. Michael  Kosterlitz  2016


In [158]:
df_last['share'] = df_last['share'].astype(int)

max_share = df_last['share'].max()
print(f"Maximum share value: {max_share}")
print(f"Smallest share value: 1/{max_share}")

print(df_last[df_last['share'] == max_share][['firstname', 'surname', 'year', 'category']].head())


Maximum share value: 4
Smallest share value: 1/4
   firstname   surname  year   category
4   Philippe    Aghion  2025  economics
5      Peter    Howitt  2025  economics
15     Demis  Hassabis  2024  chemistry
16      John    Jumper  2024  chemistry
54    Joshua   Angrist  2021  economics


In [ ]:
winner_counts = df_last['id'].value_counts()

# Let's see the top of this list
print(winner_counts.head())

id
482    3
515    2
743    2
222    2
66     2
Name: count, dtype: int64 



In [163]:
repeat_ids = winner_counts[winner_counts > 1].index

repeat_winners_df = df_last[df_last['id'].isin(repeat_ids)]

print(repeat_winners_df[['id', 'firstname', 'surname', 'year', 'category']].sort_values('id'))

       id                                          firstname    surname  year  \
687   217                                              Linus    Pauling  1962   
740   217                                              Linus    Pauling  1954   
512   222                                          Frederick     Sanger  1980   
711   222                                          Frederick     Sanger  1958   
676   482           International Committee of the Red Cross        NaN  1963   
811   482           International Committee of the Red Cross        NaN  1944   
937   482           International Committee of the Red Cross        NaN  1917   
503   515  Office of the United Nations High Commissioner...        NaN  1981   
742   515  Office of the United Nations High Commissioner...        NaN  1954   
958     6                                              Marie      Curie  1911   
1011    6                                              Marie      Curie  1903   
605    66                   

In [164]:
category_counts = df_last.groupby('id')['category'].nunique()

multi_category_ids = category_counts[category_counts > 1].index

multi_cat_winners = df_last[df_last['id'].isin(multi_category_ids)]

print("Winners in multiple categories:")
print(multi_cat_winners[['firstname', 'surname', 'category', 'year']].sort_values('surname'))

Winners in multiple categories:
     firstname  surname   category  year
958      Marie    Curie  chemistry  1911
1011     Marie    Curie    physics  1903
687      Linus  Pauling      peace  1962
740      Linus  Pauling  chemistry  1954
